<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment2/DNN_Assignment_2_DenseNet201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, datasets
from tensorflow.keras.utils import to_categorical
import numpy as np
from tensorflow.keras.applications import DenseNet201


(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train, y_train = x_train[:1000], y_train[:1000]
x_test, y_test = x_test[:200], y_test[:200]
x_train = np.stack([x_train]*3, axis=-1).astype("float32") / 255.0
x_test = np.stack([x_test]*3, axis=-1).astype("float32") / 255.0
def resize_batch(x):
    return tf.image.resize(x, [224, 224]).numpy()

x_train_resized = np.concatenate([resize_batch(x_train[i:i+500]) for i in range(0, len(x_train), 500)])
x_test_resized = np.concatenate([resize_batch(x_test[i:i+500]) for i in range(0, len(x_test), 500)])

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)
activations = ['relu', 'softmax', 'tanh', 'sigmoid']
base_model = DenseNet201(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

results = {}

for act in activations:
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation=act),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train_resized, y_train, epochs=2, batch_size=32, verbose=0,
                        validation_data=(x_test_resized, y_test))

    acc = history.history['val_accuracy'][-1]
    results[act] = acc
    print(f"Activation: {act:8s} -> Validation Accuracy: {acc:.4f}")

print("\nSummary of activation performance on DenseNet201:")
for act, acc in results.items():
    print(f"{act:<10}: {acc:.4f}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Activation: relu     -> Validation Accuracy: 0.9050
Activation: softmax  -> Validation Accuracy: 0.6800
Activation: tanh     -> Validation Accuracy: 0.8750
Activation: sigmoid  -> Validation Accuracy: 0.8400

Summary of activation performance on DenseNet201:
relu      : 0.9050
softmax   : 0.6800
tanh      : 0.8750
sigmoid   : 0.8400
